# Tacotron2-VAE — Simplified Demo for visualization and debugging



In [1]:
import os
import sys
import json
import time
import glob
from pathlib import Path
from torch.utils.data import DataLoader
from IPython.display import display, clear_output
from tqdm import tqdm

import torch
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "src" / "models" / "tacotron2_vae"))
sys.path.insert(0, str(PROJECT_ROOT / "src" / "training" / "training-tacotron2-vae"))
sys.path.insert(0, str(PROJECT_ROOT / "src" / "data" / "loader_vae_tacotron"))

from models.tacotron2_vae.hparams import create_hparams
from models.tacotron2_vae.model import load_tacotron2_vae_model, get_model_size_info
from losses import Tacotron2LossVAE
from text_processing import TextProcessor
import loader_tacotron
from hparams import Tacotron2VAEHparams
from utils import TextMelCollate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Project root: {PROJECT_ROOT}")

In [2]:
processor = TextProcessor()

dataset = loader_tacotron.DatasetLibriSpeechTacotronVAE(text_processor=processor)

print(f"Dataset size: {len(dataset)}")


In [3]:
hparams = Tacotron2VAEHparams()
model = load_tacotron2_vae_model(hparams, device=device)
criterion = Tacotron2LossVAE(hparams)
optimizer = torch.optim.Adam(model.parameters(), lr=hparams.learning_rate, weight_decay=hparams.weight_decay)

print(get_model_size_info(model))

In [4]:
# ==========================================
# 1. SETUP DE DIRETÓRIOS E CHECKPOINTS
# ==========================================
from train_utils import train_epoch, load_checkpoint, find_latest_checkpoint
from utils import create_dataloader
from loader_tacotron import load_data

exp_dir = Path("notebooks/experiments/vae_tacotron")
ckpt_dir = exp_dir / "checkpoints"
ckpt_dir.mkdir(parents=True, exist_ok=True)

hparams.experiment_dir = str(exp_dir)
hparams.checkpoint_dir = str(ckpt_dir)

# Configuração de datasets e dataloaders seguindo o pipeline oficial
train_dataset, test_dataset, val_dataset = load_data(
    text_processor=processor,
    data_dir=Path("data/processed/libriSpeech-en-tacotron-vae"),
    val_split=0.1,
    generator=torch.Generator().manual_seed(hparams.seed)
)

collate_fn = TextMelCollate(hparams.n_frames_per_step)

train_loader = create_dataloader(train_dataset, hparams.batch_size, 0, collate_fn, True)
test_loader = create_dataloader(test_dataset, hparams.batch_size, 0, collate_fn, False)
val_loader = create_dataloader(val_dataset, hparams.batch_size, 0, collate_fn, False)

iteration = 0
learning_rate = hparams.learning_rate

# Carregamento do Checkpoint Automático
try:
    latest_ckpt = find_latest_checkpoint(ckpt_dir)
    if latest_ckpt:
        print(f"[*] Checkpoint encontrado! Retomando de: {latest_ckpt}")
        model, optimizer, learning_rate, iteration = load_checkpoint(latest_ckpt, model)
        print(f"[*] Treino retomado a partir do passo {iteration}.")
except FileNotFoundError:
    print("[*] Nenhum checkpoint encontrado. Iniciando treino do zero.")

# ==========================================
# 2. LOOP DE TREINAMENTO (Pipeline Oficial)
# ==========================================
training_metadata = {
    "training_loss": [],
    "test_loss": [],
    "grad_norm": [],
    "learning_rate": [],
    "duration": [],
    "recon_loss": [],
    "kl_loss": [],
    "kl_weight": [],
    "singular_values_of_latent_covariance": [],
    "target_predict_example": [],
}

torch.backends.cudnn.enabled = hparams.cudnn_enabled
torch.backends.cudnn.benchmark = hparams.cudnn_benchmark

model.train()
print("Iniciando treinamento...")

for epoch in range(hparams.epochs):
    print(f"\n--- Epoch {epoch} ---")
    training_metadata = train_epoch(
        model=model,
        hparams=hparams,
        train_loader=train_loader,
        test_loader=test_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        iteration=iteration,
        learning_rate=learning_rate,
        training_metadata=training_metadata
    )
    # Atualiza a iteração baseada no tamanho do loader, assim como no fluxo contínuo
    iteration += len(train_loader)

print(f"\n[✔] Treinamento finalizado! Resultados e Gráficos salvos em: {exp_dir}")
